# lassoCV

This notebook demonstrates how to conduct Lasso with stratified K fold cross validation
on the Calling Cards data.  

## Pulling the data

The calling cards data should now strictly be taken from data source 'brent_nf_cc'. All
of the Mitra data has been reprocessed through the nf-core/callingcards:1.0.0 pipeline.  

Where there are multiple replicates, they have been aggregated. The `deduplicate`
parameter to `PromoterSetSigAPI()` selects aggregated data where it exists. Where
there is a single passing replicate, that replicate is used.

## Setup

As usual, import the `yeastdnnexplorer` interface functions

In [ ]:
# configure the logger to print to console
import logging

import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV

from yeastdnnexplorer.interface import PromoterSetSigAPI, ExpressionAPI, metric_arrays, rank_transforms
from yeastdnnexplorer.ml_models.lasso_modeling import (
    ModelingInputData,
    BootstrappedModelingInputData,
    stratification_classification,
    bootstrap_stratified_cv_modeling,
    evaluate_interactor_significance)


logging.basicConfig(level=logging.INFO)

pss_api = PromoterSetSigAPI()
expression_api = ExpressionAPI()

## Pull the deduplicated calling cards data

This will pull all of the currently usable data. In the future, we will remove 
"unreviewed". This will take a minute or two as it will need to fetch all of the
underlying data

In [1]:
pss_api.push_params(
    {
        "source_name": "brent_nf_cc",
        "deduplicate": "true",
        "data_usable": ["unreviewed", "pass"],
    }
)

pss_res = await pss_api.read(retrieve_files=True)

NameError: name 'pss_api' is not defined

## Pull the corresponding perturbation data

In this case, we are pulling the McIsaac data. In order to label blacklisted genes,
we'll need the shrunken data. For modeling, we will use the unshrunken data.

In [ ]:
expression_api.push_params(
    {
        "regulator_symbol": ",".join(
            pss_res.get("metadata").regulator_symbol.unique().tolist()
        ),
        "source_name": "mcisaac_oe",
        "time": "15",
    }
)

expression_res_shrunken = await expression_api.read(retrieve_files=True)

# this will add the effect_colname parameter to the expression API
expression_api.push_params(
    {
        "effect_colname": "log2_ratio",
    }
)

expression_res_unshrunken = await expression_api.read(retrieve_files=True)

## Transform the data into a usable format for modeling

Note that there are new functions, `metric_arrays` and'
`transform_scores`. See the API section of this
documentation for more details.  

You will likely want to save the results of this cell so that you do not have to run
the DB or transformation steps in future sessions, unless of course you need or want
to update the your data.

### Extract the data into a more managable format using `metric_arrays()`

In [ ]:
X = metric_arrays(
    pss_res,
    {"poisson_pval": np.min, "callingcards_enrichment": np.max},
)

Y = metric_arrays(
    expression_res_unshrunken,
    {"effect": np.max},
)

Y_shrunken = metric_arrays(
    expression_res_shrunken,
    {"effect": np.max},
)

INFO:yeastdnnexplorer.interface.metric_arrays:casting `id` to str to extract data from res_dict['data']
INFO:yeastdnnexplorer.interface.metric_arrays:casting `id` to str to extract data from res_dict['data']; Column name 'MET4' already exists in output DataFrame for metric 'effect'. Renaming to 'MET4_rep2'; Column name 'RDS2' already exists in output DataFrame for metric 'effect'. Renaming to 'RDS2_rep2'; Column name 'GZF3' already exists in output DataFrame for metric 'effect'. Renaming to 'GZF3_rep2'; Column name 'CBF1' already exists in output DataFrame for metric 'effect'. Renaming to 'CBF1_rep2'; Column name 'DAL80' already exists in output DataFrame for metric 'effect'. Renaming to 'DAL80_rep2'; Column name 'GCN4' already exists in output DataFrame for metric 'effect'. Renaming to 'GCN4_rep2'
INFO:yeastdnnexplorer.interface.metric_arrays:casting `id` to str to extract data from res_dict['data']; Column name 'MET4' already exists in output DataFrame for metric 'effect'. Renaming t

### Create gene level filters

In this case, we wish to keep only the genes common to both the binding and expression,
and create a 'blacklist' of genes which are either always responsive, or always
unresponsive. The binding and response data will then be filtered such that only the
common set of genes, which are not in the blacklist, are retained for analysis

In [ ]:
# define a set of common genes between X and y
common_genes = X["poisson_pval"].index.intersection(
    Y.get("effect", pd.DataFrame()).index
)

# binarize the Y.get("effect") DataFrame as True if the value is not 0
# We wish to exclude any genes that are always unresponsive OR always responsive
Y_binary = Y_shrunken.get("effect", pd.DataFrame()).eq(0)

always_unresponsive = Y_binary[~Y_binary.any(axis=1)].index
always_responsive = Y_binary[Y_binary.all(axis=1)].index

# combine always unresponsive and always responsive and intersect with common_genes
# to get the blacklisted genes

# define blacklisted genes as those records where the gene is either always responsive,
# or always unresponsive, in all experiments
# ensure that the callingcards engineered loci are also blacklisted
blacklisted_genes = (Y_binary[Y_binary.all(axis=1) | ~Y_binary.any(axis=1)]
                     .index.intersection(common_genes)
                     .union(['URA3', 'HIS3', 'MRM1', 'LEU2','unknown_6326']))

# Count summary
# NOTE: The blacklist is only over common genes. the always_unresponsive and always_responsive
# are over all genes in the response data
counts = pd.DataFrame.from_dict({
    "category": ["Always unresponsive (perturbation only)", "Always responsive (perturbation only)", "Blacklisted(only shared genes)"],
    "count": [len(always_unresponsive), len(always_responsive), len(blacklisted_genes)]
})

print(counts)

                                  category  count
0  Always unresponsive (perturbation only)      0
1    Always responsive (perturbation only)    140
2           Blacklisted(only shared genes)    143


### Use the common gene set and blacklist to filter the response and predictor data

TODO: ranking and transforms should be done AFTER removing the perturbed TF from the
rows

In [ ]:
# remove the blacklist genes from Y and retain only common genes
Y_filtered = Y.get("effect", pd.DataFrame()).loc[common_genes].drop(blacklisted_genes)

# remove the blacklisted_genes for X and retain only the common genes
X_filtered = {}
for key in X.keys():
    X_filtered[key] = X[key].loc[common_genes].drop(blacklisted_genes)

# Next, transform the X object into a predictors_df using the shifted negative log rank
# transformation. See `transform` for more
# details
scores_list = [
    rank_transforms.transform(
        X_filtered["poisson_pval"].loc[:, i],
        X_filtered["callingcards_enrichment"].loc[:, i],
    )
    for i in X_filtered["poisson_pval"].columns
]

# Convert the list of scores into a DataFrame
predictors_df = pd.DataFrame(scores_list).T

# Set the index and columns to match X_filtered["poisson_pval"]
predictors_df.index = X_filtered["poisson_pval"].index
predictors_df.columns = X_filtered["poisson_pval"].columns


# conduct a similar shifted negative log rank transformation on the Y values
Y_filtered_ranked = Y_filtered.rank(ascending=False, method="average")
Y_filtered_transformed = Y_filtered_ranked.apply(
    rank_transforms.shifted_negative_log_ranks, axis=0)

## Modeling per TF

This demonstrates the usage on a single TF. The TFs are the columns in both the 
response and predictor dataframes. To run this on all TFs, you would simply iterate
over the columns of the response DF. This can be done in parallel very easily on the
cluster.

INFO:main:Response column names: Index(['CBF1'], dtype='object')
INFO:main:Common features between response and predictors: 6007. Subsetting and reordering both dataframes.
INFO:main:Number of blacklisted features: 0
INFO:main:Selected 597 top features based on descending ranking of predictors_df['CBF1'].
INFO:main:Top-n feature masking enabled.
INFO:main:Top-n feature masking disabled.


In [ ]:
input_data = ModelingInputData(
    response_df=Y_filtered_transformed.loc[:, ['CBF1']].reset_index(names="target_symbol"),
    predictors_df=predictors_df.reset_index(names="target_symbol"),
    perturbed_tf="CBF1",
    feature_blacklist=list(blacklisted_genes),
    top_n = 600,
)

input_data.top_n_masked = False

predictor_variables = input_data.predictors_df.columns.drop(
        input_data.perturbed_tf
    )

interaction_terms = [
        f"{input_data.perturbed_tf}:{var}" for var in predictor_variables
    ]

# Construct the formula as a single expression (no intercept)
formula = f"{input_data.perturbed_tf} + {' + '.join(interaction_terms)} - 1"

# add input_data.perturbed_tf squared to the formula
formula += f" + I({input_data.perturbed_tf} ** 2)"

# add the row_max term to the formula
formula += " + row_max"



In [ ]:

#n_bootstraps=1000,

import importlib
from yeastdnnexplorer.ml_models import lasso_modeling

importlib.reload(lasso_modeling)

all_data_indicies = lasso_modeling.BootstrappedModelingInputData.load_indices("/home/chase/code/yeastdnnexplorer/tmp/bootstrapped_data_all_indices.csv")

bootstrapped_data_all = lasso_modeling.BootstrappedModelingInputData(
    response_df=input_data.response_df,
    model_df = input_data.get_modeling_data(formula, add_row_max=True),
    bootstrap_indices=all_data_indicies
)



INFO:main:Using integer sample weights: True


In [ ]:


# bootstrapped_data_all.save_indices(
#     "/home/chase/code/yeastdnnexplorer/tmp/bootstrapped_data_all_indices.csv",
# )


In [ ]:

classes = stratification_classification(input_data.predictors_df[input_data.perturbed_tf].squeeze(), input_data.response_df.squeeze())


In [ ]:

# NOTE: fit_intercept is set to `true`
estimator = LassoCV(
    fit_intercept=True,
    max_iter=10000,
    selection="random",
    random_state=42,
    n_jobs=4)

perturbed_tf_series = input_data.predictors_df[
        input_data.perturbed_tf
    ]

all_data_results = bootstrap_stratified_cv_modeling(
        bootstrapped_data_all,
        perturbed_tf_series,
        estimator=estimator,
        ci_percentiles=[98.0],
        use_sample_weight_in_cv=True,
    )


INFO:main:Using sample weights in CV: True
INFO:main:Response frame shape: (6006, 1)
INFO:main:Model frame shape: (6006, 122)
INFO:main:Model frame columns: Index(['CBF1', 'CBF1:DOT6', 'CBF1:MTH1', 'CBF1:FZF1', 'CBF1:PIP2', 'CBF1:HAA1',
       'CBF1:SFL1', 'CBF1:XBP1', 'CBF1:WTM1', 'CBF1:SWI4',
       ...
       'CBF1:GAL80', 'CBF1:CIN5', 'CBF1:SUM1', 'CBF1:UPC2', 'CBF1:ROX1',
       'CBF1:RPN4', 'CBF1:CSE2', 'CBF1:LEU3', 'I(CBF1 ** 2)', 'row_max'],
      dtype='object', length=122)
INFO:main:Using binning strategy: binding and perturbation
INFO:main:Using the following stratification bins: [0, 8, 64, 512, inf].
INFO:main:Starting bootstrap modeling iterations...


In [ ]:
all_data_sig_coefs = all_data_results.extract_significant_coefficients(
    ci_level="98.0"
)

all_data_sig_coefs

{'CBF1:DAT1': (-0.14286967641924053, -0.018873395684263595),
 'CBF1:MET28': (0.04002889873744846, 0.15271750992733288),
 'CBF1:GIS1': (0.021641096885868856, 0.20174064282021303),
 'row_max': (0.03860610260032976, 0.0831032369627654)}

In [ ]:
-0.14286967641924064-(-0.14286967641924053)

-1.1102230246251565e-16

In [ ]:
'''
{'CBF1:DAT1': (-0.14286967641924064, -0.018873395684263595),
 'CBF1:MET28': (0.04002889873744846, 0.15271750992733288),
 'CBF1:GIS1': (0.021641096885868856, 0.20174064282021303),
 'row_max': (0.03860610260032976, 0.0831032369627654)}
'''

input_data.top_n_masked = True

topn_formula = f"{' + '.join(all_data_sig_coefs.keys())}"

topn_formula

'CBF1:DAT1 + CBF1:MET28 + CBF1:RPH1 + CBF1:GIS1 + CBF1:AFT1 + CBF1:GAL4'

In [ ]:

bootstrapped_data_top_n = BootstrappedModelingInputData(
    response_df=input_data.response_df,
    model_df = input_data.get_modeling_data(topn_formula),
    n_bootstraps=1000,
)

tf_series_top_n = input_data.predictors_df[
        input_data.perturbed_tf
    ]

topn_classes = stratification_classification(tf_series_top_n.squeeze(), input_data.response_df.squeeze())

In [ ]:
topn_results = bootstrap_stratified_cv_modeling(
    bootstrapped_data_top_n,
    perturbed_tf_series=tf_series_top_n,
    estimator=estimator,
    ci_percentiles=[90.0],
    use_sample_weight_in_cv=False,
)

In [ ]:
topn_results.extract_significant_coefficients(ci_level = "90.0")

{'CBF1:DAT1': (-0.15028976904756874, -0.029657540547558096),
 'CBF1:MET28': (0.07272477583603269, 0.1621220234890702)}

In [ ]:
input_data.top_n_masked = False

alldata_classes = stratification_classification(
    input_data.predictors_df[input_data.perturbed_tf].squeeze(),
    input_data.response_df.squeeze())

results = evaluate_interactor_significance(
        input_data,
        stratification_classes=alldata_classes,
        model_variables=list(topn_results.extract_significant_coefficients(ci_level = "90.0").keys()),
    )

/home/chase/code/yeastdnnexplorer/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(
/home/chase/code/yeastdnnexplorer/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(
/home/chase/code/yeastdnnexplorer/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(
/home/chase/code/yeastdnnexplorer/.venv/lib/python3.11/site-packages/sklearn/model_selection/_split.py:737: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=4.
  warnings.warn(


In [ ]:
print(results.data)

[{'interactor': 'CBF1:DAT1',
  'variant': 'DAT1',
  'avg_r2_interactor': 0.020282642711779958,
  'avg_r2_main_effect': 0.018492734456674764,
  'delta_r2': -0.0017899082551051937},
 {'interactor': 'CBF1:MET28',
  'variant': 'MET28',
  'avg_r2_interactor': 0.020282642711779958,
  'avg_r2_main_effect': 0.011762906645376742,
  'delta_r2': -0.008519736066403216}]

In [ ]:
print(results.final_model())

## Visualize and extract significant coefficients

You can use `examine_boostrap_coefficients()` to generate a plot, and extract a list,
of coeffiecents whose user specified ci_interval exists beyond a specified threshold. 
For example, if `ci_interval = 95.0` and `threshold = 0`, then the coefficients that
are returned are those whose 95.0 confidence interval do not cross zero.  

In the example below, I set the ci_interval to 100.0 with a threshold of 0, which 
will return only coefficients where none of the bootstrap values cross 0.

In [ ]:
# Let's say we want to return coefficients where none of the bootstrapped coefficient
# estimates include 0. We can leave the `threshold` parameter as 0.0, and set `ci_level`
# to 100.00
# res = init_results.extract_significant_coefficients(
#     threshold=0.0, ci_level=98.0
# )

# # we can also find out which coefficient has a value closest to 0
# min_key, min_bounds = min(res.items(),
#                           key=lambda item: min(abs(item[1][0]), abs(item[1][1])))

# print(f"\nThe coefficient closest to 0 is: {min_key}: {min_bounds}")

In [ ]:
# init_results.visualize_significant_coefficients(
#     threshold=0.0, ci_level=95.0
# )

## Using the cmd line utility

There is a command line utility called lasso_bootstrap that may be launched like this:

```bash
#!/bin/bash

python -m yeastdnnexplorer lasso_bootstrap \
    --response_file ../lasso_response_mcisaac15.csv \
    --predictors_file ../lasso_predictors_callingcards.csv \
    --perturbed_tf CBF1 \
    --n_bootstrap 1000 \
    --log-level ERROR
```

The output will, by default, be in a directory in the `PWD` called
`lasso_bootstrap_output` with subdirectories named by `--perturbed_tf`. The output
of `bootstrap_lasso_output()` will be saved, along with a lassoCV model run on the
entire dataset as a joblib file.  

To examine the results using `examine_bootstrap_coefficients()`, you would load the
data like this:

```python
import json
import pandas as pd

from yeastdnnexplorer.ml_models.lasso_modeling import examine_bootstrap_coefficients


with open("lasso_bootstrap_output/CBF1/ci_dict.json", "r") as file:
    coef_dict = json.load(file)

coef_df = pd.read_csv("testing_lasso_cmd/lasso_bootstrap_output/CBF1/bootstrap_coef_df.csv")

alphas_list = pd.read_csv("lasso_bootstrap_output/CBF1/bootstrap_alphas.csv")['alpha'].tolist()

bootstrap_lasso_output = (coef_dict, coef_df, alphas_list)

sig_coef_plt, sig_coef_dict = examine_bootstrap_coefficients(
    bootstrap_lasso_output,
    ci_level=100.0)

```

## Flowchart

```mermaid
graph LR;
    %% Inputs
    subgraph Inputs
        IN1[Response file]
        IN2[Predictors file]
        IN3[Perturbed TF]
        IN4[Optional: Feature Blacklist]
        IN5[Optional: Formula]
        IN6[Optional: Output directory]
    end

    %% Inputs to Validation
    IN1 --> A
    IN2 --> A
    IN3 --> A
    IN4 --> A
    IN5 --> A
    IN6 --> A

    %% Data flow
    A[Step 1: Preprocessing] --> B[Step 2: Bootstrapped 4-fold LassoCV]
    B --> C[Reduce predictors to only significant predictors]
    C --> D[Step 3: Bootstrapped 4-fold LassoCV on top 10%]
    D --> E[Reduce predictors to only significant predictors]
    E --> F[Step 4: Test interactions against main effects]

<!-- markdownlint-disable MD013 -->
    %% Outputs
    subgraph Outputs
        OUT1@{ shape: lean-r, label: "BootstrapModelResults in /all_data_result_object" }
        OUT2@{ shape: lean-r, label: "Significant coefficients at all_data_significant_<ci_level>.json" }
        OUT3@{ shape: lean-r, label: "BootstrapModelResults in /topn_result_object" }
        OUT4@{ shape: lean-r, label: "Significant coefficients at topn_significant_<ci_level>.json`" }
        OUT5@{ shape: lean-r, label: "InteractorSignificanceResults interactor_vs_main_result.json" }
    end
<!-- markdownlint-enable MD013 -->

    %% Linking Outputs to Process
    B -.-> OUT1
    C -.-> OUT2
    D -.-> OUT3
    E -.-> OUT4
    F -.-> OUT5
```

```mermaid
graph LR;

    %% Inputs
    subgraph Inputs
        IN1[BootstrappedModelingInputData]
        IN2[Perturbed TF Series]
        IN3[Estimator]
        IN4[CI percentiles]
        IN5[Use sample weights]
        IN6[Bin by binding and perturbation]
    end

    %% Input routing
    IN1 --> A[Validation, preprocessing and initializations]
    IN2 --> A
    IN3 --> A
    IN4 --> N[Compute confidence intervals]
    IN5 --> A
    IN6 --> SS

    %% Bootstrap Loop
    A --> C[Set random_state on estimator and CV]

    subgraph "Bootstrap Loop"
        C --> D{Use sample weights?}

        %% Stratification strategy section (wrapped visually)
        subgraph SS[Stratification Strategy]
            direction TB
            E1[Generate stratification from full data]
            E2[Generate stratification from resampled data]
        end

        D -- Yes --> E1
        D -- No  --> E2

        E1 --> F1[Fit model using full data and weights]
        E2 --> F2[Fit model using resampled data only]

        F1 --> G[Store alpha and coefficients]
        F2 --> G
        G --> H{More bootstrap samples?}
        H -- Yes --> C
        H -- No --> I[Aggregate coefficients into DataFrame]
    end

    I --> N

    %% Outputs
    subgraph Outputs
        OUT1[BootstrapModelResults]
    end

    N --> OUT1
```